In [1]:
from pathlib import Path
import fitz  # PyMuPDF

document_path = Path("../data/documents/raw/nasa_rcmguide.pdf")

with fitz.open(document_path) as pdf:
    page_count = len(pdf)
    first_page_text = pdf[0].get_text()

print("Pages:", page_count)
print("\nFirst-page preview:\n")
print(first_page_text[:1000])

Pages: 472

First-page preview:

 
 
RCM GUIDE
RELIABILITY-CENTERED 
MAINTENANCE GUIDE
 For Facilities and 
Collateral Equipment
FINAL
September 2008 
DRAFT
National Aeronautics and Space Administration 



In [2]:
import json

documents_dir = Path("../data/documents")
sources_path = documents_dir / "sources.json"

with sources_path.open(encoding="utf-8") as file:
    sources = json.load(file)["sources"]


def extract_pages(source: dict) -> list[dict]:
    pdf_path = documents_dir / source["local_file"]
    extracted_pages = []

    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text().strip()

            if text:
                extracted_pages.append(
                    {
                        "source_id": source["id"],
                        "title": source["title"],
                        "source_url": source["source_url"],
                        "page_number": page_number,
                        "text": text,
                    }
                )

    return extracted_pages


pages = []

for source in sources:
    pages.extend(extract_pages(source))

print("Extracted pages:", len(pages))

for source in sources:
    source_pages = [page for page in pages if page["source_id"] == source["id"]]
    print(f"- {source['title']}: {len(source_pages)} readable pages")

Extracted pages: 515
- NASA Reliability-Centered Maintenance Guide: 471 readable pages
- Control of Hazardous Energy: Lockout/Tagout: 44 readable pages


In [3]:
def split_text(
    text: str,
    chunk_size: int = 900,
    overlap: int = 150,
) -> list[str]:
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks


chunks = []

for page in pages:
    page_chunks = split_text(page["text"])

    for chunk_number, text in enumerate(page_chunks, start=1):
        chunks.append(
            {
                "source_id": page["source_id"],
                "title": page["title"],
                "source_url": page["source_url"],
                "page_number": page["page_number"],
                "chunk_number": chunk_number,
                "text": text,
            }
        )

print("Searchable chunks:", len(chunks))
print("\nExample chunk:\n")
print(chunks[0]["text"][:1000])

Searchable chunks: 1342

Example chunk:

RCM GUIDE
RELIABILITY-CENTERED 
MAINTENANCE GUIDE
 For Facilities and 
Collateral Equipment
FINAL
September 2008 
DRAFT
National Aeronautics and Space Administration


In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Embedding matrix shape:", embeddings.shape)

D:\Stuff\Coding\SentinelAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6188.59it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Batches:   2%|▏         | 1/42 [00:01<00:43,  1.05s/it]

...

Batches:  95%|█████████▌| 40/42 [00:31<00:01,  1.69it/s]

Batches:  98%|█████████▊| 41/42 [00:31<00:00,  1.86it/s]

Batches: 100%|██████████| 42/42 [00:31<00:00,  2.42it/s]

Batches: 100%|██████████| 42/42 [00:31<00:00,  1.32it/s]

Embedding matrix shape: (1342, 384)


In [5]:
import numpy as np


def search_documents(query: str, top_k: int = 3) -> list[dict]:
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True,
    )

    similarity_scores = embeddings @ query_embedding

    top_indices = np.argsort(similarity_scores)[::-1][:top_k]

    return [
        {
            **chunks[index],
            "similarity_score": float(similarity_scores[index]),
        }
        for index in top_indices
    ]


query = "How does condition monitoring support maintenance planning?"

results = search_documents(query)

for result in results:
    print(
        f"\nScore: {result['similarity_score']:.3f}"
        f"\nSource: {result['title']}"
        f"\nPage: {result['page_number']}"
        f"\nText: {result['text'][:500]}"
    )
    print("-" * 80)


Score: 0.776
Source: NASA Reliability-Centered Maintenance Guide
Page: 239
Text: NASA RELIABILITY-CENTERED MAINTENANCE GUIDE 
FOR FACILITIES AND COLLATERAL EQUIPMENT 
 
September 2008 
A-3
Condition Monitoring (also known as Predictive Maintenance): The continuous or periodic 
monitoring and diagnosis of systems and equipment in order to forecast failure.  Condition 
Monitoring is a Time- or Cycle-Based Maintenance Action In condition monitoring advanced 
technology is used to assess machinery condition.  The data obtained allows for planning and 
scheduling preventive maint
--------------------------------------------------------------------------------

Score: 0.729
Source: NASA Reliability-Centered Maintenance Guide
Page: 17
Text: NASA RELIABILITY-CENTERED MAINTENANCE GUIDE 
FOR FACILITIES AND COLLATERAL EQUIPMENT 
 
September 2008             
2.2    CONDITION-BASED MONITORING 
By the 1980s alternatives to traditional Preventive Maintenance (PM) programs began to 
migrate to the m

In [6]:
evaluation_queries = [
    {
        "query": "When should condition monitoring intervals be shortened?",
        "expected_source_id": "nasa-rcm-guide",
    },
    {
        "query": "How can equipment condition data identify failure precursors?",
        "expected_source_id": "nasa-rcm-guide",
    },
    {
        "query": "What must be verified before maintenance begins on locked-out equipment?",
        "expected_source_id": "osha-lockout-tagout",
    },
    {
        "query": "What should happen before machinery is reenergized after maintenance?",
        "expected_source_id": "osha-lockout-tagout",
    },
]

hits = 0

for item in evaluation_queries:
    results = search_documents(item["query"], top_k=3)

    found_expected_source = any(
        result["source_id"] == item["expected_source_id"]
        for result in results
    )

    hits += found_expected_source

    print(f"\nQuestion: {item['query']}")
    print(f"Expected source: {item['expected_source_id']}")
    print(f"Found in top 3: {found_expected_source}")

recall_at_3 = hits / len(evaluation_queries)

print(f"\nRetrieval Recall@3: {recall_at_3:.0%}")


Question: When should condition monitoring intervals be shortened?
Expected source: nasa-rcm-guide
Found in top 3: True

Question: How can equipment condition data identify failure precursors?
Expected source: nasa-rcm-guide
Found in top 3: True

Question: What must be verified before maintenance begins on locked-out equipment?
Expected source: osha-lockout-tagout
Found in top 3: True

Question: What should happen before machinery is reenergized after maintenance?
Expected source: osha-lockout-tagout
Found in top 3: False

Retrieval Recall@3: 75%


In [7]:
for item in evaluation_queries:
    results = search_documents(item["query"], top_k=3)

    found_expected_source = any(
        result["source_id"] == item["expected_source_id"]
        for result in results
    )

    if not found_expected_source:
        print("Missed question:", item["query"])
        print("Expected source:", item["expected_source_id"])
        print("Returned sources:")

        for result in results:
            print(
                f"- {result['source_id']} "
                f"(score: {result['similarity_score']:.3f})"
            )

Missed question: What should happen before machinery is reenergized after maintenance?
Expected source: osha-lockout-tagout
Returned sources:
- nasa-rcm-guide (score: 0.593)
- nasa-rcm-guide (score: 0.588)
- nasa-rcm-guide (score: 0.546)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
)

tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)


def scale_to_zero_one(scores: np.ndarray) -> np.ndarray:
    score_range = scores.max() - scores.min()

    if score_range == 0:
        return np.ones_like(scores)

    return (scores - scores.min()) / score_range


def expand_safety_query(query: str) -> str:
    safety_terms = (
        "reenerg",
        "deenerg",
        "lockout",
        "tagout",
        "isolate",
        "hazardous energy",
    )

    if any(term in query.lower() for term in safety_terms):
        return (
            f"{query} lockout tagout hazardous energy "
            "energy control procedure"
        )

    return query


def search_documents(query: str, top_k: int = 3) -> list[dict]:
    search_query = expand_safety_query(query)
    query_embedding = embedding_model.encode(
        search_query,
        normalize_embeddings=True,
    )

    semantic_scores = embeddings @ query_embedding

    query_tfidf = tfidf_vectorizer.transform([search_query])
    keyword_scores = cosine_similarity(
        query_tfidf,
        tfidf_matrix,
    ).flatten()

    combined_scores = (
        0.5 * scale_to_zero_one(semantic_scores)
        + 0.5 * keyword_scores
    )

    top_indices = np.argsort(combined_scores)[::-1][:top_k]

    return [
        {
            **chunks[index],
            "similarity_score": float(combined_scores[index]),
        }
        for index in top_indices
    ]

In [9]:
hits = 0

for item in evaluation_queries:
    results = search_documents(item["query"], top_k=3)
    found_expected_source = any(
        result["source_id"] == item["expected_source_id"]
        for result in results
    )
    hits += found_expected_source
    print(f"{item['expected_source_id']}: {found_expected_source}")

recall_at_3 = hits / len(evaluation_queries)
print(f"\nHybrid Retrieval Recall@3: {recall_at_3:.0%}")


nasa-rcm-guide: True
nasa-rcm-guide: True
osha-lockout-tagout: True
osha-lockout-tagout: True

Hybrid Retrieval Recall@3: 100%
